# GRB006 nidq event repair

Manual DB repair notebook for `GRB006` `20240821_121447`, dataset `ephys_g0`, stream `nidq`.

Guardrails:
- leave `nidq:0` unchanged
- rebuild behavioral rows `1..5` as `io3`, `io2`, `io4`, `io5`, `io6`
- insert visual `io0` from local `trial_ts.pkl`
- do not insert `io1` yet; it is reserved for future audio work

Intended final `nidq` rows for this session:
- `0`
- `io0`
- `io2`
- `io3`
- `io4`
- `io5`
- `io6`

Current GRB006 interpretation:
- `nidq:2` behaves like trial-start TTL with paired on/off edges and will become `io2`
- `nidq:1` is treated as `io3` (frames)


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from labdata.schema import DatasetEvents

SUBJECT = "GRB006"
SESSION = "20240821_121447"
DATASET = "ephys_g0"
STREAM = "nidq"
TRIAL_TS_PATH = Path("/Users/gabriel/Downloads/Organized/Code/trial_ts.pkl")

NUMERIC_TO_CANONICAL = {
    "1": "io3",
    "2": "io2",
    "3": "io4",
    "4": "io5",
    "5": "io6",
}
EXPECTED_EVENT_NAMES = ["0", "io0", "io2", "io3", "io4", "io5", "io6"]
RESTRICTION = {
    "subject_name": SUBJECT,
    "session_name": SESSION,
    "dataset_name": DATASET,
    "stream_name": STREAM,
}

In [ ]:
def fetch_stream_rows() -> pd.DataFrame:
    rows = pd.DataFrame((DatasetEvents.Digital() & RESTRICTION).fetch(as_dict=True))
    if rows.empty:
        raise RuntimeError(f"No DatasetEvents.Digital rows found for {RESTRICTION}")
    rows = rows.copy()
    rows["n_events"] = rows["event_timestamps"].apply(
        lambda ts: 0 if ts is None else len(np.asarray(ts))
    )
    rows["head_s"] = rows["event_timestamps"].apply(
        lambda ts: (
            [] if ts is None else np.asarray(ts, dtype=float)[:5].round(6).tolist()
        )
    )
    return rows.sort_values("event_name").reset_index(drop=True)


def show_stream_rows() -> pd.DataFrame:
    cols = ["event_name", "n_events", "head_s"]
    display_df = fetch_stream_rows()[cols]
    try:
        from IPython.display import display

        display(display_df)
    except Exception:
        print(display_df.to_string(index=False))
    return display_df


def load_visual_onsets(trial_ts_path: Path = TRIAL_TS_PATH) -> np.ndarray:
    if not trial_ts_path.exists():
        raise FileNotFoundError(f"Missing trial_ts file: {trial_ts_path}")
    trial_ts = pd.read_pickle(trial_ts_path).reset_index(drop=True)
    if "stim_ts_visual" not in trial_ts.columns:
        raise RuntimeError("trial_ts.pkl does not contain stim_ts_visual")
    visual_stims = [
        np.asarray(stims, dtype=float)
        for stims in trial_ts["stim_ts_visual"].tolist()
        if len(stims)
    ]
    if not visual_stims:
        raise RuntimeError("No visual stim timestamps found in trial_ts.pkl")
    all_visual = np.concatenate(visual_stims)
    all_visual = all_visual[np.isfinite(all_visual)]
    if all_visual.size == 0:
        raise RuntimeError("No finite visual stim timestamps found in trial_ts.pkl")
    return np.sort(all_visual)


def build_replacement_row(old_event_name: str, new_event_name: str) -> dict:
    old_key = {**RESTRICTION, "event_name": old_event_name}
    row = (DatasetEvents.Digital() & old_key).fetch1()
    return {
        **RESTRICTION,
        "event_name": new_event_name,
        "event_timestamps": np.asarray(row["event_timestamps"], dtype=float),
        "event_values": None
        if row["event_values"] is None
        else np.asarray(row["event_values"]),
    }


def insert_or_replace_row(row: dict) -> None:
    key = {k: row[k] for k in RESTRICTION.keys()} | {"event_name": row["event_name"]}
    rel = DatasetEvents.Digital() & key
    if len(rel):
        rel.delete(force=True)
    DatasetEvents.Digital().insert1(row, allow_direct_insert=True)


def repair_behavior_rows() -> None:
    for old_event_name, new_event_name in NUMERIC_TO_CANONICAL.items():
        replacement = build_replacement_row(old_event_name, new_event_name)
        insert_or_replace_row(replacement)
    old_rows = DatasetEvents.Digital() & [
        {**RESTRICTION, "event_name": name} for name in NUMERIC_TO_CANONICAL
    ]
    if len(old_rows):
        old_rows.delete(force=True)


def insert_visual_io0() -> np.ndarray:
    visual_onsets = load_visual_onsets()
    io0_row = {
        **RESTRICTION,
        "event_name": "io0",
        "event_timestamps": visual_onsets,
        "event_values": np.ones(visual_onsets.shape[0], dtype=np.uint8),
    }
    insert_or_replace_row(io0_row)
    return visual_onsets


def fetch_chipmunk_trial_count() -> int:
    from chipmunk import Chipmunk

    restriction = f'subject_name = "{SUBJECT}" AND session_name = "{SESSION}"'
    trial_df = pd.DataFrame((Chipmunk * Chipmunk.Trial & restriction).fetch())
    return len(trial_df)


def verify_end_state() -> pd.DataFrame:
    rows = fetch_stream_rows()
    event_names = rows["event_name"].tolist()
    expected_present = set(EXPECTED_EVENT_NAMES)
    found = set(event_names)
    missing = sorted(expected_present - found)
    unexpected_io = sorted(
        name for name in found if name.startswith("io") and name not in expected_present
    )
    if missing:
        raise AssertionError(f"Missing expected event rows: {missing}")
    if "io1" in found:
        raise AssertionError("Found io1, but io1 should remain absent for now")
    if unexpected_io:
        raise AssertionError(f"Unexpected io rows present: {unexpected_io}")
    return rows

## Current DB state

Inspect the current `nidq` rows before touching anything.

In [ ]:
show_stream_rows();

## Local visual source

Load visual onset timestamps from the local `trial_ts.pkl` source that will become `nidq:io0`.

In [ ]:
visual_onsets = load_visual_onsets()
print(f"Loaded {len(visual_onsets)} visual onset timestamps from {TRIAL_TS_PATH}")
print("First 10 onsets (s):", np.round(visual_onsets[:10], 6))

## Repair behavioral rows

This rebuilds `1..5` as `io2..io6` and deletes the old numeric behavioral rows. `nidq:0` is untouched.

In [ ]:
repair_behavior_rows()
show_stream_rows();

## Insert visual `io0`

This inserts or replaces `nidq:io0` using the trusted local visual timestamps.

In [ ]:
visual_onsets = insert_visual_io0()
print(f"Inserted nidq:io0 with {len(visual_onsets)} visual events")
show_stream_rows();

## Verify end state

Confirm the notebook ends with exactly `0`, `io0`, `io2..io6` on `nidq`, with no `io1`.

In [ ]:
from IPython.display import display

final_rows = verify_end_state()
display(final_rows[["event_name", "n_events", "head_s"]])
print("Final event names:", final_rows["event_name"].tolist())

## Optional trial-count sanity check

Compare the repaired `io2` trial-start count against Chipmunk trial count.

In [ ]:
io2_ts = np.asarray(
    (DatasetEvents.Digital() & {**RESTRICTION, "event_name": "io2"}).fetch1(
        "event_timestamps"
    ),
    dtype=float,
)
chipmunk_trials = fetch_chipmunk_trial_count()
io2_rising = io2_ts[::2] if io2_ts.size else io2_ts
print(f"Chipmunk trials: {chipmunk_trials}")
print(
    f"io2 timestamps: {len(io2_ts)} total edges, {len(io2_rising)} rising-edge starts"
)